### 1. Bibliotecas

In [1]:
import pandas as pd
import numpy as np

import os
import warnings
import itertools

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_log_error, mean_squared_error, mean_absolute_error
from sklearn.metrics import make_scorer, mean_squared_log_error
import joblib

from sklearn.feature_selection import mutual_info_regression

In [2]:
# Definindo diretório
os.chdir("G:\\Meu Drive\\MeuDrive2\\academico\\3.kaggle\\2.Predict_Calorie_Expenditure")

# Exibir todas as linhas e colunas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Não exibir avisos
warnings.filterwarnings("ignore")

### 2. Importação e separação dos dados

In [3]:
# Importando os dados
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_submission = pd.read_csv('sample_submission.csv')

In [4]:
# Verificando se há dados faltantes
train.isna().sum()

id            0
Sex           0
Age           0
Height        0
Weight        0
Duration      0
Heart_Rate    0
Body_Temp     0
Calories      0
dtype: int64

In [5]:
# Separando os dados
from sklearn.model_selection import train_test_split

target = "Calories"
X = train.drop(columns=[target])
y = train[target]

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, # Conjunto de validação
                                                      random_state=42, 
                                                      shuffle=True)

In [6]:
X_train.head()

,id,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp
453635,453635,male,43,190.0,89.0,6.0,87.0,39.1
11651,11651,female,48,155.0,54.0,12.0,97.0,40.2
431999,431999,male,51,187.0,92.0,15.0,96.0,40.5
529211,529211,male,45,182.0,88.0,2.0,83.0,38.3
110925,110925,male,22,202.0,99.0,25.0,98.0,40.7


### 3. Criação de variáveis

#### 3.1 Faixa_Etaria

In [7]:
# Criar categorias com base na idade
bins_faixa_etaria = [0, 30, 55, float('inf')]  # limites: 0–30, 31–55, 56+
labels_faixa_etaria = ['Jovem', 'Adulto', 'Idoso']

X_train['Faixa_Etaria'] = pd.cut(X_train['Age'], bins=bins_faixa_etaria, labels=labels_faixa_etaria, right=True, include_lowest=True)
X_valid['Faixa_Etaria'] = pd.cut(X_valid['Age'], bins=bins_faixa_etaria, labels=labels_faixa_etaria, right=True, include_lowest=True)

#### 3.2 IMC e IMC_classificacao

In [8]:
# Criando a coluna IMC
X_train['IMC'] = X_train['Weight'] / ((X_train['Height'] / 100) ** 2)
X_valid['IMC'] = X_valid['Weight'] / ((X_valid['Height'] / 100) ** 2)

In [9]:
# Função que categoriza o IMC em magro, normal, sobrepeso, obeso_1, obse_2 e obseo_3
def classificar_imc(imc):
    if imc <= 18.5:
        return 'magro'
    elif 18.6 <= imc <= 24.9:
        return 'normal'
    elif 25.0 <= imc <= 29.9:
        return 'sobrepeso'
    elif 30.0 <= imc <= 34.9:
        return 'obeso_1'
    elif 35.0 <= imc <= 39.9:
        return 'obeso_2'
    else:
        return 'obeso_3'

In [10]:
X_train['IMC_classificacao'] = X_train['IMC'].apply(classificar_imc)
X_valid['IMC_classificacao'] = X_valid['IMC'].apply(classificar_imc)

#### 3.3 calories_per_min e total_calories

In [11]:
# Um atributo Sex_male deve ser criada antes
X_train['Sex_male'] = (X_train['Sex'] == 'male').astype(int)
X_valid['Sex_male'] = (X_valid['Sex'] == 'male').astype(int)

In [12]:
def calculate_calories(df):
    # Fórmula para homens
    cal_men = (-55.0969 + 0.6309 * df['Heart_Rate'] + 0.1988 * df['Weight'] + 0.2017 * df['Age']) / 4.184
    
    # Fórmula para mulheres
    cal_women = (-20.4022 + 0.4472 * df['Heart_Rate'] - 0.1263 * df['Weight'] + 0.074 * df['Age']) / 4.184
    
    # Aplica a fórmula correta com base no sexo
    df['calories_per_min'] = np.where(df['Sex_male'] == 1, cal_men, cal_women)
    
    # Calcula as calorias totais pela duração
    df['total_calories'] = df['calories_per_min'] * df['Duration']
    
    return df

In [13]:
X_train = calculate_calories(X_train)
X_valid = calculate_calories(X_valid)

#### 3.4 TMB e TDEE

In [14]:
def calculate_tmb_naf(df, naf=1.55):
    # TMB homens
    tmb_men = 66 + (13.7 * df['Weight']) + (5 * df['Height']) - (6.8 * df['Age'])
    # TMB mulheres
    tmb_women = 655 + (9.6 * df['Weight']) + (1.8 * df['Height']) - (4.7 * df['Age'])
    
    # Escolhe a fórmula certa por sexo
    df['TMB'] = np.where(df['Sex_male'] == 1, tmb_men, tmb_women)
    
    # Aplica o nível de atividade física
    df['TDEE'] = df['TMB'] * naf  # TDEE = Total Daily Energy Expenditure
    
    return df

In [15]:
# Aplica nas bases
X_train = calculate_tmb_naf(X_train)
X_valid = calculate_tmb_naf(X_valid)

#### 3.6 Categorizando Duaration

In [16]:
# Definindo os bins e os nomes das categorias
bins_duration = [0, 10, 20, float('inf')]
labels_duration = ['duracao_curta', 'duracao_media', 'duracao_longa']

# Criando a nova coluna categórica
X_train['Duration_cat'] = pd.cut(X_train['Duration'], bins=bins_duration, labels=labels_duration, right=True)
X_valid['Duration_cat'] = pd.cut(X_valid['Duration'], bins=bins_duration, labels=labels_duration, right=True)

#### 3.7 Categorizar Body_Temp

In [17]:
# Criar coluna categórica
X_train['Body_Temp_cat'] = np.where(X_train['Body_Temp'] <= 39.6, 'baixa', 'normal')
X_valid['Body_Temp_cat'] = np.where(X_valid['Body_Temp'] <= 39.6, 'baixa', 'normal')

#### 3.8 Categorizando Heart_Rate

In [18]:
bins_heart_rate = [35, 54, 69, 89, float('inf')]
labels_heart_rate = ['muito_leve', 'leve', 'moderada', 'alta']

X_train['Heart_Rate_cat'] = pd.cut(X_train['Heart_Rate'], bins=bins_heart_rate, labels=labels_heart_rate, right=True)
X_valid['Heart_Rate_cat'] = pd.cut(X_valid['Heart_Rate'], bins=bins_heart_rate, labels=labels_heart_rate, right=True)

#### 3.9 HeartRate_per_kg  e Duration_per_weight

- Variáveis obtidas em https://www.kaggle.com/code/bhaskarmishra44796/calorie-expenditure-xgbreg pelo usuário Bhaskar (Mishra bhaskarmishra44796) 

In [19]:
# HeartRate_per_kg
X_train['HeartRate_per_kg'] = X_train['Heart_Rate'] / X_train['Weight']
X_valid['HeartRate_per_kg'] = train['Heart_Rate'] / X_valid['Weight']

In [20]:
# Duration_per_weight
X_train['Duration_per_weight']=X_train['Duration']/X_train['Weight']
X_valid['Duration_per_weight']=X_valid['Duration']/X_valid['Weight']

#### 3.10 Heart_Stress e heart_rate_per_duration

- variáveis obtidas em https://www.kaggle.com/code/khsamaha/calories-predictions-eda-xgboost-catboost-r pelo usuário Kheirallah Samaha (khsamaha)

In [21]:
# heart_stress
X_train['Heart_Stress'] = X_train['Heart_Rate']*X_train['Duration']
X_valid['Heart_Stress'] = X_valid['Heart_Rate']*X_valid['Duration']

In [22]:
# heart_rate_per_duration
X_train['heart_rate_per_duration'] = X_train['Heart_Rate'] / X_train['Duration']
X_valid['heart_rate_per_duration'] = X_valid['Heart_Rate'] / X_valid['Duration']

#### 3.11 Dummies

In [23]:
# Criando dummies

# Aplicando get_dummies nas novas categorias
X_train_dummies = pd.get_dummies(X_train, columns=['Faixa_Etaria', 'IMC_classificacao', 'Duration_cat', 'Body_Temp_cat', 'Heart_Rate_cat'], drop_first=True)

X_valid_dummies = pd.get_dummies(X_valid, columns=['Faixa_Etaria', 'IMC_classificacao', 'Duration_cat', 'Body_Temp_cat', 'Heart_Rate_cat'], drop_first=True)

# Reindexando X_valid para ter as mesmas colunas que X_train
X_valid_dummies = X_valid_dummies.reindex(columns=X_train_dummies.columns, fill_value=0)

# Convertendo booleanos para inteiros (caso necessário)
for column in X_train_dummies.columns:
    if X_train_dummies[column].dtype == 'bool':
        X_train_dummies[column] = X_train_dummies[column].astype(int)
        X_valid_dummies[column] = X_valid_dummies[column].astype(int)

In [24]:
X_train_dummies = X_train_dummies.drop(['id', 'Sex'], axis=1)
X_valid_dummies = X_valid_dummies.drop(['id', 'Sex'], axis=1)

#### 4. Outliers

In [25]:
# Lista de variáveis a tratar
variaveis = X_train_dummies.select_dtypes(include=['float', 'float64']).columns.tolist()

# Dicionários para armazenar limites e medianas
limites = {}
medianas = {}

In [26]:
# 1. Calcular IQR e mediana com base em X_train
for var in variaveis:
    Q1 = X_train_dummies[var].quantile(0.25)
    Q3 = X_train_dummies[var].quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    mediana = X_train_dummies[var].median()

    limites[var] = (limite_inferior, limite_superior)
    medianas[var] = mediana

In [27]:
# 2. Função para substituir outliers
def substituir_outliers(df, limites, medianas):
    df_corrigido = df.copy()
    for var in variaveis:
        inf, sup = limites[var]
        med = medianas[var]
        df_corrigido[var] = df_corrigido[var].apply(lambda x: med if x < inf or x > sup else x)
    return df_corrigido

In [28]:
# 3. Aplicar aos conjuntos
X_train_dummies = substituir_outliers(X_train_dummies, limites, medianas)
X_valid_dummies = substituir_outliers(X_valid_dummies, limites, medianas)

#### 5. Padronizando os dados

In [29]:
# Padronizando os dados
from sklearn.preprocessing import MinMaxScaler

# Inicializando o escalonador Min-Max
scaler = MinMaxScaler()

# Escalonando os dados de treino e validação
X_train_scaled = scaler.fit_transform(X_train_dummies)
X_valid_scaled = scaler.transform(X_valid_dummies)

# Convertendo os resultados de volta para DataFrame
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train_dummies.columns)
X_valid_scaled = pd.DataFrame(X_valid_scaled, columns=X_valid_dummies.columns)

### 6. Selecionando atributos

#### 6.1 Constantes

In [30]:
# Verificando se há atributos constantes
constant_features = [
    feat for feat in X_train_scaled.columns if X_train_scaled[feat].std() == 0
]

print(constant_features)

[]


#### 6.2 Quasi-Constantes

In [31]:
# create an empty list
quasi_constant_feat = []

# iterate over every feature
for feature in X_train_scaled.columns:

    # find the predominant value, that is the value that is shared
    # by most observations
    predominant = X_train_scaled[feature].value_counts(
        normalize=True).sort_values(ascending=False).values[0]

    # evaluate the predominant feature: do more than 99% of the observations
    # show 1 value?
    if predominant > 0.998:

        # if yes, add the variable to the list
        quasi_constant_feat.append(feature)

len(quasi_constant_feat)

3

In [32]:
print(quasi_constant_feat)

['IMC_classificacao_obeso_1', 'IMC_classificacao_obeso_2', 'Heart_Rate_cat_leve']


In [33]:
# Remover essas variáveis
X_train_scaled = X_train_scaled.drop(['IMC_classificacao_obeso_1', 'IMC_classificacao_obeso_2', 'Heart_Rate_cat_leve'], axis=1)

X_valid_scaled = X_valid_scaled.drop(['IMC_classificacao_obeso_1', 'IMC_classificacao_obeso_2', 'Heart_Rate_cat_leve'], axis=1)

#### 6.3 Atributos suplicados

In [34]:
# check for duplicated features in the training set:

# create an empty dictionary, where we will store 
# the groups of duplicates
duplicated_feat_pairs = {}

# create an empty list to collect features
# that were found to be duplicated
_duplicated_feat = []


# iterate over every feature in our dataset:
for i in range(0, len(X_train_scaled.columns)):
    
    # this bit helps me understand where the loop is at:
    if i % 10 == 0:  
        print(i)
    
    # choose 1 feature:
    feat_1 = X_train_scaled.columns[i]
    
    # check if this feature has already been identified
    # as a duplicate of another one. If it was, it should be stored in
    # our _duplicated_feat list.
    
    # If this feature was already identified as a duplicate, we skip it, if
    # it has not yet been identified as a duplicate, then we proceed:
    if feat_1 not in _duplicated_feat:
    
        # create an empty list as an entry for this feature in the dictionary:
        duplicated_feat_pairs[feat_1] = []

        # now, iterate over the remaining features of the dataset:
        for feat_2 in X_train_scaled.columns[i + 1:]:

            # check if this second feature is identical to the first one
            if X_train_scaled[feat_1].equals(X_train_scaled[feat_2]):

                # if it is identical, append it to the list in the dictionary
                duplicated_feat_pairs[feat_1].append(feat_2)
                
                # and append it to our monitor list for duplicated variables
                _duplicated_feat.append(feat_2)
                
                # done!

0
10
20


In [35]:
# let's explore our list of duplicated features
len(_duplicated_feat)

0

### 6.4 Seleção final dos atributos

In [36]:
X_train_scaled.head()

,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,IMC,Sex_male,calories_per_min,total_calories,TMB,TDEE,HeartRate_per_kg,Duration_per_weight,Heart_Stress,heart_rate_per_duration,Faixa_Etaria_Adulto,Faixa_Etaria_Idoso,IMC_classificacao_normal,IMC_classificacao_obeso_3,IMC_classificacao_sobrepeso,Duration_cat_duracao_media,Duration_cat_duracao_longa,Body_Temp_cat_normal,Heart_Rate_cat_moderada,Heart_Rate_cat_alta
0,0.389831,0.696203,0.609195,0.172414,0.344828,0.314286,0.531752,1.0,0.499329,0.124730,0.584748,0.584748,0.245032,0.100579,0.120594,0.681818,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.474576,0.253165,0.206897,0.379310,0.517241,0.628571,0.287272,0.0,0.367944,0.188620,0.161096,0.161096,0.781568,0.366083,0.290750,0.310606,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0
2,0.525424,0.658228,0.643678,0.482759,0.500000,0.714286,0.717629,1.0,0.660015,0.408837,0.568006,0.568006,0.288248,0.264587,0.363901,0.213223,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0
3,0.423729,0.594937,0.597701,0.034483,0.275862,0.085714,0.746581,1.0,0.452089,0.036970,0.544933,0.544933,0.222525,0.023934,0.026239,0.202479,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4,0.033898,0.848101,0.724138,0.827586,0.534483,0.771429,0.487799,1.0,0.594912,0.618106,0.785778,0.785778,0.253138,0.418055,0.631593,0.069752,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0


In [37]:
# 1. Calcular matriz de correlação entre as features
corr_matrix = X_train_scaled.corr().abs()

# 2. Calcular correlação de cada variável com o target
target_corr = X_train_scaled.apply(lambda col: np.corrcoef(col, y_train)[0, 1])

# 3. Encontrar pares de variáveis altamente correlacionadas
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# 4. Remover uma variável de cada par altamente correlacionado (baseado na menor correlação com o target)
to_drop = set()

threshold = 0.95  # seu limiar de correlação

for col in upper_tri.columns:
    high_corr = upper_tri[col][upper_tri[col] > threshold]
    for row in high_corr.index:
        if target_corr[col] >= target_corr[row]:
            to_drop.add(row)
        else:
            to_drop.add(col)

# 5. Filtrar os conjuntos de dados
selected_features = [col for col in X_train_scaled.columns if col not in to_drop]

X_train_selected = X_train_scaled[selected_features]
X_valid_selected = X_valid_scaled[selected_features]
#X_test_selected  = X_test_scaled[selected_features]

print("Variáveis selecionadas:", selected_features)

Variáveis selecionadas: ['Age', 'Weight', 'Heart_Rate', 'Body_Temp', 'IMC', 'Sex_male', 'calories_per_min', 'total_calories', 'TMB', 'HeartRate_per_kg', 'Duration_per_weight', 'Heart_Stress', 'heart_rate_per_duration', 'Faixa_Etaria_Adulto', 'Faixa_Etaria_Idoso', 'IMC_classificacao_normal', 'IMC_classificacao_obeso_3', 'IMC_classificacao_sobrepeso', 'Duration_cat_duracao_media', 'Duration_cat_duracao_longa', 'Body_Temp_cat_normal', 'Heart_Rate_cat_alta']


In [38]:
X_train_selected.columns == X_valid_selected.columns

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True])

### 7. Previsão

In [ ]:
"""import xgboost as xgb - Melhor modelo até agora

# Modelo base
xgb_model = xgb.XGBRegressor(random_state=42, objective='reg:squarederror')

# Grade de parâmetros ajustada
param_grid = {
    'n_estimators': [100, 300, 500, 700],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.03, 0.05, 0.1]
}

# Busca em grade com validação cruzada usando RMSLE
grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=5,
    scoring='neg_root_mean_squared_log_error',
    verbose=3
)

# Treinamento
grid_search.fit(X_train_selected, y_train)

# Melhor modelo
best_xgb_model = grid_search.best_estimator_
y_pred = best_xgb_model.predict(X_valid_selected)

# Avaliação manual de RMSLE (para garantir)
rmsle = np.sqrt(mean_squared_log_error(y_valid, np.maximum(0, y_pred)))  # garante não negativo

# Resultados
print("Melhores parâmetros:", grid_search.best_params_)
print(f"RMSLE: {rmsle:.4f}")"""

In [40]:
import xgboost as xgb

# Novo modelo com os melhores parâmetros
final_xgb_model = xgb.XGBRegressor(
    learning_rate=0.03,
    max_depth=9,
    n_estimators=700,
    random_state=42,
    objective='reg:squarederror'
)

# Treinamento com todo o conjunto de dados de treino (ou com X_train + X_valid)
final_xgb_model.fit(X_train_selected, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.03, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=9,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=700,
             n_jobs=None, num_parallel_tree=None, ...)

### 8. teste

In [41]:
# Copiar e prepara X_test
X_test = test.copy()
test_ids = X_test["id"]

In [42]:
bins_faixa_etaria = [0, 30, 55, float('inf')]
labels = ['Jovem', 'Adulto', 'Idoso']  # Reafirme que são 3 labels

X_test['Faixa_Etaria'] = pd.cut(
    X_test['Age'],
    bins=bins_faixa_etaria,
    labels=labels_faixa_etaria,
    right=True,
    include_lowest=True
)

In [43]:
# Criando a coluna IMC
X_test['IMC'] = X_test['Weight'] / ((X_test['Height'] / 100) ** 2)

# Classifica IMC
X_test['IMC_classificacao'] = X_test['IMC'].apply(classificar_imc)

In [44]:
X_test['Sex_male'] = (X_test['Sex'] == 'male').astype(int)

# calories_per_min e total_calories
X_test = calculate_calories(X_test)

# TMB e TDEE
X_test = calculate_tmb_naf(X_test)

In [45]:
# Criando a nova coluna categórica
X_test['Duration_cat'] = pd.cut(X_test['Duration'], bins=bins_duration, labels=labels_duration, right=True)

In [46]:
# Criar coluna categórica
X_test['Body_Temp_cat'] = np.where(X_test['Body_Temp'] <= 39.6, 'baixa', 'normal')

In [47]:
X_test['Heart_Rate_cat'] = pd.cut(X_test['Heart_Rate'], bins=bins_heart_rate, labels=labels_heart_rate, right=True)

In [48]:
# HeartRate_per_kg e Duration_per_weight
X_test['HeartRate_per_kg'] = X_test['Heart_Rate'] / X_test['Weight']
X_test['Duration_per_weight']=X_test['Duration']/X_test['Weight']

In [49]:
# heart_stress e heart_rate_per_duration
X_test['Heart_Stress'] = X_test['Heart_Rate']*X_test['Duration']
X_test['heart_rate_per_duration'] = X_test['Heart_Rate'] / X_test['Duration']

In [50]:
# Gerar dummies (mesmas colunas de treino)
X_test_dummies = pd.get_dummies(X_test, columns=['Faixa_Etaria', 'IMC_classificacao', 'Duration_cat', 'Body_Temp_cat', 'Heart_Rate_cat'], drop_first=True)

# Reindexar para ter as mesmas colunas do treino
X_test_dummies = X_test_dummies.reindex(columns=X_train_dummies.columns, fill_value=0)

# Garantir booleanos como int
for column in X_test_dummies.columns:
    if X_test_dummies[column].dtype == 'bool':
        X_test_dummies[column] = X_test_dummies[column].astype(int)

In [51]:
# Outliers (Aprendidos em medianas)
X_test_dummies = substituir_outliers(X_test_dummies, limites, medianas)

In [52]:
# padronizando os dados
# Escalonar X_test_dummies com o mesmo scaler treinado em X_train_dummies
X_test_scaled = scaler.transform(X_test_dummies)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test_dummies.columns)

In [53]:
# removendo variáveis quasi constantes
X_test_scaled = X_test_scaled.drop(['IMC_classificacao_obeso_1', 'IMC_classificacao_obeso_2', 'Heart_Rate_cat_leve'], axis=1)

In [54]:
# Se quiser aplicar ao teste:
X_test_selected  = X_test_scaled[selected_features]

In [55]:
# Verificando se possuem as mesmas colunas
X_test_selected.columns == X_train_selected.columns

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True])

In [56]:
# previsão
y_test_pred = final_xgb_model.predict(X_test_selected)

In [57]:
# Submissão

# importa o arquivo de submissão
submission = pd.read_csv("sample_submission.csv")

# Substituis os valores da coluna calories pelos valores preditos
submission["Calories"] = y_test_pred

# Garante que os ids estão corretos
submission["id"] = test_ids 

# Exporta a submissão
submission.to_csv("final_xgb.csv", index=False)